# Week 3 Day 5: Coding Agent 设计与 SGLang 面试故事

今天的目标不是再写一个复杂 Agent，而是把 SWE-agent 的 ACI 思想、Devin 式长任务工作流、OWASP LLM 安全风险，整理成能在面试里讲清楚的 SGLang 项目故事。

这个 Notebook 和 `agent.py` 保持同一套内容：脚本负责结构化生成，Notebook 负责逐段理解和演练。

## 1. 加载今天的故事生成器

目录名里有连字符，不能直接 `import exercises.w3d5-coding-agent-story.agent`，所以这里用 `importlib` 从文件路径加载。

In [ ]:
from pathlib import Path
import importlib.util
import sys

AGENT_PATH = Path("agent.py")
if not AGENT_PATH.exists():
    AGENT_PATH = Path("exercises/w3d5-coding-agent-story/agent.py")

spec = importlib.util.spec_from_file_location("coding_agent_story", AGENT_PATH)
story = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = story
spec.loader.exec_module(story)

pack = story.build_story_pack()
len(pack.principles), len(pack.failure_modes), len(pack.rehearsal_prompts)

## 2. Coding Agent 设计原则

面试时不要只说“AI 帮我写代码更快”。更有区分度的表达是：你知道 coding agent 靠什么工作、它的边界在哪里、你如何把它放进可验证的工程闭环。

In [ ]:
for principle in pack.principles:
    print(f"## {principle.name} ({principle.source_anchor})")
    print("面试表达:", principle.interview_point)
    print("SGLang 映射:", principle.sglang_mapping)
    print()

## 3. STAR 主线

`Situation -> Task -> Action -> Result` 是故事骨架。SGLang 这段经历的重点是：复杂代码库、清晰 spec、分阶段 review、测试反馈和人类架构判断。

In [ ]:
for section in pack.star_story:
    print(section.title)
    for bullet in section.bullets:
        print(" -", bullet)
    print()

## 4. 失败模式

好的项目故事必须能讲失败模式。这里只保留三类最像真实工程的追问：过度抽象、边界 case 漏测、性能直觉不足，再补一个 agent 权限边界问题。

In [ ]:
for failure in pack.failure_modes:
    print(f"风险: {failure.risk}")
    print(f"例子: {failure.example}")
    print(f"控制: {failure.control}")
    print()

## 5. Agent 安全速记

OWASP LLM Top 10 不需要在这里逐条背成安全考试，但要能把风险映射到 coding agent 的真实工作流。

In [ ]:
for note in pack.safety_notes:
    print("-", note)

## 6. 90 秒背诵版

这一段可以直接作为面试开场回答。后续追问再展开 spec、trace、测试、失败模式。

In [ ]:
print(pack.pitch_90s)
print("\n字数:", len(pack.pitch_90s))

## 7. 生成 Markdown 大纲

`STORY.md` 是今天的正式产出，可以打开单独背诵，也可以后续继续补真实 PR、benchmark 和 review 例子。

In [ ]:
markdown = story.render_markdown(pack)
output_path = AGENT_PATH.parent / "STORY.md"
output_path.write_text(markdown, encoding="utf-8")
print("written:", output_path)
print(markdown[:1000])

## 8. 自检

脚本里的 `validate_story_pack` 用来防止大纲退化成空泛总结：它要求 90 秒版本包含 task spec、trace、测试、review 和 SGLang 等关键词，也要求有足够的失败模式与安全提醒。

In [ ]:
errors = story.validate_story_pack(pack)
assert not errors, errors
print("story pack validation passed")